# Импрорты и нужные классы

In [ ]:
import sys
import os
import torch
import copy
import numpy as np
from dotenv import load_dotenv

sys.path.append('..')
env_file = os.path.join('..', '.env')
load_dotenv(env_file)
rowGlam_type = os.environ['ROW_GLAM_TYPE'] # custom or base
# name_dataset = os.environ['NAME_DATASET']
# name_test_dataset = os.environ['NAME_TEST_DATASET']
# dataset_path = os.environ['DATASET_PATH']
# test_path = os.environ['TEST_PATH']
# test_coco_path = os.environ['TEST_COCO_PATH']
# coco_path = os.environ['COCO_PATH']
# cache_pdf = os.environ['CASH_PDF_PATH']

doclaynet_train_pdf_path = os.environ['DOCLAYNET_TRAIN_PDF_PATH']
doclaynet_coco_path = os.environ['DOCLAYNET_COCO_PATH']
doclaynet_test_pdf_path = os.environ['DOCLAYNET_TEST_PDF_PATH']
doclaynet_test_coco_path_for_doc = os.environ['DOCLAYNET_TEST_COCO_PATH_FOR_DOC']
#запустить скрипт doc2pub для создания coco аннотации с метками как в publaynet
doclaynet_test_coco_path_for_pub = os.environ['DOCLAYNET_TEST_COCO_PATH_FOR_PUB']
doclaynet_train_cash_pdf_path = os.environ['DOCLAYNET_TRAIN_CASH_PDF_PATH']
doclaynet_test_cash_pdf_path_for_doc = os.environ['DOCLAYNET_TEST_CASH_PDF_PATH_FOR_DOC']
doclaynet_test_cash_pdf_path_for_pub = os.environ['DOCLAYNET_TEST_CASH_PDF_PATH_FOR_PUB']

publaynet_train_pdf_path = os.environ['PUBLAYNET_TRAIN_PDF_PATH']
publaynet_coco_path = os.environ['PUBLAYNET_COCO_PATH']
publaynet_test_pdf_path = os.environ['PUBLAYNET_TEST_PDF_PATH']
publaynet_test_coco_path = os.environ['PUBLAYNET_TEST_COCO_PATH']
publaynet_train_cash_pdf_path = os.environ['PUBLAYNET_TRAIN_CASH_PDF_PATH']
publaynet_test_cash_pdf_path = os.environ['PUBLAYNET_TEST_CASH_PDF_PATH']

In [ ]:
from rows2regionsGLAM.utils.pdf_manager import PDFManager
from rows2regionsGLAM.utils.loger import Loger
from rows2regionsGLAM.utils.row_manager import RowManager
from rows2regionsGLAM.utils.ploter import Ploter
from rows2regionsGLAM.utils.trainer import Trainer
from rows2regionsGLAM.utils.tester import Tester
from rows2regionsGLAM.utils.cacher import Cacher
from rows2regionsGLAM.utils.coco_manager import COCOManager
from rows2regionsGLAM.utils.imbalance import calculate_imbalance
from rows2regionsGLAM.utils.tester import collect_maps, print_map_table

from rows2regionsGLAM.models.rowGLAM_base import TorchModelBase, PARAMS_BASE
from rows2regionsGLAM.models.rowGLAM_custom import TorchModel, PARAMS

from rows2regionsGLAM.tokenizers import RowGLAMTokenizer
from rows2regionsGLAM.converters import Rows2Regions
from rows2regionsGLAM.datasetloaders.base_line_dataset import GLAMDataset
from pager.page_model.sub_models import RegionModel, RowsModel

In [ ]:
loger = Loger('log.txt')
pdf_manager = PDFManager(conf={"loger": loger, "pdf_reader": "PDFMiner"})
row_manager = RowManager(conf={"loger": loger, "add_image": True})

pub_coco_manager = COCOManager(conf={"loger": loger, "coco_path": publaynet_coco_path})
doc_coco_manager = COCOManager(conf={"loger": loger, "coco_path": doclaynet_coco_path})

ploter = Ploter(conf={"loger": loger})
tokenizer = RowGLAMTokenizer()
loger(tokenizer.get_name())

pdf2torch_dict = Cacher({
    "loger": loger,
    "pdf_manager": pdf_manager,
    "row_manager": row_manager,
    "tokenizer": tokenizer
})

# PubLayNet

In [ ]:
json_true_regions, PUB_CLASSES = pub_coco_manager.get_regions_from_json()

In [ ]:
PUB_CLASSES[3] = "text"
PUB_CLASSES

In [ ]:
file_names = list(json_true_regions.keys())
file_names.sort()

In [ ]:
# отрисовка, проверка работы менеджеров
num_file = 1
file_name = file_names[num_file]
dataset_file = os.path.join(publaynet_train_pdf_path, file_name)
pdf_json, pdf_img = pdf_manager.get_json_and_img_from_pdf(dataset_file, num_page=0)
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)
torch_dict = tokenizer(row_json, pdf_img)
ploter.set_dpi(200)
ploter.plot_img(pdf_img)
ploter.plot_tokens(tokenizer, torch_dict)

## Сохранение датасета

In [ ]:
pub_dataset = GLAMDataset({
    "loger": loger,
    "pdf_dir": publaynet_train_pdf_path,
    "coco_file": publaynet_coco_path,
    "count_class": len(PUB_CLASSES),
    "name_dataset": "publaynet",
    "default_index": 0,
    "cache_dir": publaynet_train_cash_pdf_path,
    "pdf2torch_dict": pdf2torch_dict
})

# Создание Cache
N = len(pub_dataset)
for i, d in enumerate(pub_dataset):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

## Отрисовска файла из датасета

In [ ]:
torch_dict_pub_train = pub_dataset[1050]
path_pdf = os.path.join(publaynet_train_pdf_path, torch_dict_pub_train['file_name'] + ".pdf")
_, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
ploter.plot_tokens(tokenizer, torch_dict_pub_train, markup=True)

## Параметры модели

In [ ]:
PUB_GLAM_MODEL = 'pub_row2region_GLAM_artic'
if rowGlam_type == "base":
    PUB_PARAMS = copy.deepcopy(PARAMS_BASE)
elif rowGlam_type == "custom":
    PUB_PARAMS = copy.deepcopy(PARAMS)
    
PUB_PARAMS["NodeClasses"] = len(PUB_CLASSES)

PUB_PARAMS["epochs"] = 30
PUB_PARAMS["batch_size"] = 128

if rowGlam_type == "custom":
    PUB_PARAMS["model_type"] = 2 # 1 or 2 or 3

In [ ]:
# Дисбаланс классов ------------------------------------
publaynet_imbalance, edge_imbalance = calculate_imbalance(pub_dataset)

print(publaynet_imbalance, edge_imbalance)

PUB_PARAMS['loss_params']['publaynet_imbalance'] = publaynet_imbalance
PUB_PARAMS['loss_params']['edge_imbalance'] = edge_imbalance

PUB_PARAMS

## Обучение модели

In [ ]:
# model:torch.nn.Module = TorchModel(PUB_PARAMS)
# total_params = sum(p.numel() for p in model.parameters())  
# print(f"Number of parameters: {total_params}") 

In [ ]:
trainer_pub = Trainer(conf={"loger": loger, "params": PUB_PARAMS, "model_name": PUB_GLAM_MODEL})
trainer_pub.start_train(5, pub_dataset)

# DocLayNet

In [ ]:
json_true_regions, DOC_CLASSES = doc_coco_manager.get_regions_from_json()

In [ ]:
# DOC_CLASSES[3] = "text"
DOC_CLASSES

In [ ]:
file_names = list(json_true_regions.keys())
file_names.sort()

In [ ]:
# отрисовка, проверка работы менеджеров
num_file = 1
file_name = file_names[num_file]
dataset_file = os.path.join(doclaynet_train_pdf_path, file_name)
pdf_json, pdf_img = pdf_manager.get_json_and_img_from_pdf(dataset_file, num_page=0)
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)
torch_dict = tokenizer(row_json, pdf_img)
ploter.set_dpi(200)
ploter.plot_img(pdf_img)
ploter.plot_tokens(tokenizer, torch_dict)

## Сохранение датасета

In [ ]:
doc_dataset = GLAMDataset({
    "loger": loger,
    "pdf_dir": doclaynet_train_pdf_path,
    "coco_file": doclaynet_coco_path,
    "count_class": len(DOC_CLASSES),
    "name_dataset": "doclaynet",
    "default_index": 0,
    "cache_dir": doclaynet_train_cash_pdf_path,
    "pdf2torch_dict": pdf2torch_dict
})

# Создание Cache
N = len(doc_dataset)
for i, d in enumerate(doc_dataset):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

## Отрисовска файла из датасета

In [ ]:
torch_dict_doc_train = doc_dataset[100]
path_pdf = os.path.join(doclaynet_train_pdf_path, torch_dict_doc_train['file_name'] + ".pdf")
_, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
ploter.plot_tokens(tokenizer, torch_dict_doc_train, markup=True)

## Параметры модели

In [ ]:
DOC_GLAM_MODEL = 'doc_row2region_GLAM_artic'
if rowGlam_type == "base":
    DOC_PARAMS = copy.deepcopy(PARAMS_BASE)
elif rowGlam_type == "custom":
    DOC_PARAMS = copy.deepcopy(PARAMS)
    
DOC_PARAMS["NodeClasses"] = len(DOC_CLASSES)

DOC_PARAMS["epochs"] = 30
DOC_PARAMS["batch_size"] = 128

if rowGlam_type == "custom":
    DOC_PARAMS["model_type"] = 2 # 1 or 2 or 3

In [ ]:
# Дисбаланс классов ------------------------------------
publaynet_imbalance, edge_imbalance = calculate_imbalance(doc_dataset)

print(publaynet_imbalance, edge_imbalance)

DOC_PARAMS['loss_params']['publaynet_imbalance'] = publaynet_imbalance
DOC_PARAMS['loss_params']['edge_imbalance'] = edge_imbalance

DOC_PARAMS

## Обучение модели

In [ ]:
# model:torch.nn.Module = TorchModel(PUB_PARAMS)
# total_params = sum(p.numel() for p in model.parameters())  
# print(f"Number of parameters: {total_params}") 

In [ ]:
trainer_doc = Trainer(conf={"loger": loger, "params": DOC_PARAMS, "model_name": DOC_GLAM_MODEL})
trainer_doc.start_train(5, doc_dataset)

# Проверка работы модели PubLayNet

In [ ]:
PUB_PARAMS['sigmoidEdge'] = True

if rowGlam_type == "base":
    model_pub = TorchModelBase(PUB_PARAMS)
elif rowGlam_type == "custom":
    model_pub = TorchModel(PUB_PARAMS)

model_pub.load_state_dict(torch.load(PUB_GLAM_MODEL, weights_only=True))

rows_model = RowsModel()
region_model = RegionModel()
rows2regions_pub = Rows2Regions({
    'model':model_pub, 
    'tokenizer': RowGLAMTokenizer(),
    'is_merge_extract': True,
    'classes': PUB_CLASSES
})

In [ ]:
torch_dict_pub_train = pub_dataset[100]
path_pdf = os.path.join(publaynet_train_pdf_path, torch_dict_pub_train['file_name'] + ".pdf")
pdf_json, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
# ploter.plot_tokens(tokenizer, torch_dict_pub_train)

# Регионы --------------------------------------------------------------
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)

rows_model.from_dict({"rows": row_json})
rows2regions_pub.convert(rows_model, region_model, img)

for reg in region_model.regions:
    reg.segment.plot(text=reg.label)

# Проверка работы модели DocLayNet

In [ ]:
DOC_PARAMS['sigmoidEdge'] = True

if rowGlam_type == "base":
    model_doc = TorchModelBase(DOC_PARAMS)
elif rowGlam_type == "custom":
    model_doc = TorchModel(DOC_PARAMS)

model_doc.load_state_dict(torch.load(DOC_GLAM_MODEL, weights_only=True))

rows_model = RowsModel()
region_model = RegionModel()
rows2regions_doc = Rows2Regions({
    'model':model_doc, 
    'tokenizer': RowGLAMTokenizer(),
    'is_merge_extract': True,
    'classes': DOC_CLASSES
})

In [ ]:
torch_dict_doc_train = doc_dataset[3000]
path_pdf = os.path.join(doclaynet_train_pdf_path, torch_dict_doc_train['file_name'] + ".pdf")
pdf_json, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
# ploter.plot_tokens(tokenizer, torch_dict_doc_train)

# Регионы --------------------------------------------------------------
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)

rows_model.from_dict({"rows": row_json})
rows2regions_doc.convert(rows_model, region_model, img)

for reg in region_model.regions:
    reg.segment.plot(text=reg.label)

# Тестирование 

## Создание тестовых датасетов

### publaynet test dataset 

In [ ]:
pub_test_dataset = GLAMDataset(
    {
    "loger": loger,
    "pdf_dir": publaynet_test_pdf_path,
    "coco_file": publaynet_test_coco_path,
    "count_class": len(PUB_CLASSES),
    "name_dataset": "publaynet",
    "default_index": 0,
    "cache_dir": publaynet_test_cash_pdf_path,
    "pdf2torch_dict": pdf2torch_dict
    }
)

torch_dict2 = pub_test_dataset[100]
path_pdf = os.path.join(publaynet_test_pdf_path, torch_dict2['file_name'] + ".pdf")
pdf_json, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
# ploter.plot_tokens(tokenizer, torch_dict2)

# Регионы --------------------------------------------------------------
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)

rows_model.from_dict({"rows": row_json})
rows2regions_pub.convert(rows_model, region_model, img)

for reg in region_model.regions:
    reg.segment.plot(text=reg.label)

In [ ]:
loger("Start Test")
loger.time_log()

N = len(pub_test_dataset)
for i, d in enumerate(pub_test_dataset):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

### doclaynet test dataset for doclaynet

In [ ]:
doc_test_dataset_for_doc = GLAMDataset(
    {
    "loger": loger,
    "pdf_dir": doclaynet_test_pdf_path,
    "coco_file": doclaynet_test_coco_path_for_doc,
    "count_class": len(DOC_CLASSES),
    "name_dataset": "doclaynet",
    "default_index": 0,
    "cache_dir": doclaynet_test_cash_pdf_path_for_doc,
    "pdf2torch_dict": pdf2torch_dict
    }
)

torch_dict2 = doc_test_dataset_for_doc[50]
path_pdf = os.path.join(doclaynet_test_pdf_path, torch_dict2['file_name'] + ".pdf")
pdf_json, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
# ploter.plot_tokens(tokenizer, torch_dict2)

# Регионы --------------------------------------------------------------
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)

rows_model.from_dict({"rows": row_json})
rows2regions_doc.convert(rows_model, region_model, img)

for reg in region_model.regions:
    reg.segment.plot(text=reg.label)

In [ ]:
loger("Start Test")
loger.time_log()

N = len(doc_test_dataset_for_doc)
for i, d in enumerate(doc_test_dataset_for_doc):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

### doclaynet test dataset for publaynet

In [ ]:
doc_test_dataset_for_pub = GLAMDataset(
    {
    "loger": loger,
    "pdf_dir": doclaynet_test_pdf_path,
    "coco_file": doclaynet_test_coco_path_for_pub,
    "count_class": len(DOC_CLASSES),
    "name_dataset": "doclaynet",
    "default_index": 0,
    "cache_dir": doclaynet_test_cash_pdf_path_for_pub,
    "pdf2torch_dict": pdf2torch_dict
    }
)

torch_dict2 = doc_test_dataset_for_pub[50]
path_pdf = os.path.join(doclaynet_test_pdf_path, torch_dict2['file_name'] + ".pdf")
pdf_json, img = pdf_manager.get_json_and_img_from_pdf(path_pdf)
ploter.set_dpi(200)
ploter.plot_img(img)
# ploter.plot_tokens(tokenizer, torch_dict2)

# Регионы --------------------------------------------------------------
row_json = row_manager.get_row_json_from_pdf_json(pdf_json)

rows_model.from_dict({"rows": row_json})
rows2regions_doc.convert(rows_model, region_model, img)

for reg in region_model.regions:
    reg.segment.plot(text=reg.label)

In [ ]:
loger("Start Test")
loger.time_log()

N = len(doc_test_dataset_for_pub)
for i, d in enumerate(doc_test_dataset_for_pub):
    print(f"{(i+1)/N*100:4.2f} %", end='\r')

## Оценка

In [ ]:
loger_train_pub_test_pub = Loger('log_train_pub_test_pub.txt')
loger_train_pub_test_doc = Loger('log_train_pub_test_doc.txt')
loger_train_doc_test_doc = Loger('log_train_doc_test_doc.txt')
loger_train_doc_test_pub = Loger('log_train_doc_test_pub.txt')

### Train pub Test pub

In [ ]:
tester_train_pub_test_pub = Tester(conf={
    "loger": loger_train_pub_test_pub, 
    "pdf_manager": pdf_manager, 
    "row_manager": row_manager, 
    "rows_model": rows_model, 
    "region_model": region_model, 
    "rows2regions": rows2regions_pub})

metrics_pp = tester_train_pub_test_pub.calculate_target_and_preds(
    pub_test_dataset, 
    "publaynet", 
    "publaynet", 
    publaynet_train_pdf_path, 
    publaynet_test_pdf_path
)

### Train pub Test doc

In [ ]:
tester_train_pub_test_doc = Tester(conf={
    "loger": loger_train_pub_test_doc, 
    "pdf_manager": pdf_manager, 
    "row_manager": row_manager, 
    "rows_model": rows_model, 
    "region_model": region_model, 
    "rows2regions": rows2regions_pub})

metrics_pd = tester_train_pub_test_doc.calculate_target_and_preds(
    doc_test_dataset_for_pub, 
    "publaynet", 
    "doclaynet", 
    publaynet_train_pdf_path, 
    doclaynet_test_pdf_path
)

### Train doc Test doc

In [ ]:
tester_train_doc_test_doc = Tester(conf={
    "loger": loger_train_doc_test_doc, 
    "pdf_manager": pdf_manager, 
    "row_manager": row_manager, 
    "rows_model": rows_model, 
    "region_model": region_model, 
    "rows2regions": rows2regions_doc})

metrics_dd = tester_train_doc_test_doc.calculate_target_and_preds(
    doc_test_dataset_for_doc, 
    "doclaynet", 
    "doclaynet", 
    doclaynet_train_pdf_path, 
    doclaynet_test_pdf_path
)

### Train doc Test pub

In [ ]:
tester_train_doc_test_pub = Tester(conf={
    "loger": loger_train_doc_test_pub, 
    "pdf_manager": pdf_manager, 
    "row_manager": row_manager, 
    "rows_model": rows_model, 
    "region_model": region_model, 
    "rows2regions": rows2regions_doc})

metrics_dp = tester_train_doc_test_pub.calculate_target_and_preds(
    pub_test_dataset, 
    "doclaynet", 
    "publaynet", 
    doclaynet_train_pdf_path, 
    publaynet_test_pdf_path
)

### Вывод результатов

In [ ]:
print("train_pub_test_pub")
tester_train_pub_test_pub.print_result(metrics_pp)

print("train_pub_test_doc")
tester_train_pub_test_doc.print_result(metrics_pd)

print("train_doc_test_doc")
tester_train_doc_test_doc.print_result(metrics_dd)

print("train_doc_test_pub")
tester_train_doc_test_pub.print_result(metrics_dp)

In [ ]:
log_directory = "./"

table = collect_maps(log_directory)
print_map_table(table)

In [ ]:
mAP@IoU[0.50:0.95]   :0.52044123
==================================================
threshold_05--------------------
precision_row       :0.8745
recall_row          :0.7442
f1_row              :0.8041
precision_word      :0.8312
recall_word         :0.7130
f1_word             :0.7676
threshold_95--------------------
precision_row       :0.7275
recall_row          :0.6222
f1_row              :0.6707
precision_word      :0.6884
recall_word         :0.5949
f1_word             :0.6382